# Backpropagation in RNN

Backpropagation in Recurrent Neural Networks (RNNs) is technically called Backpropagation Through Time (BPTT).
Standard backpropagation works on Feedforward Neural Networks, which are acyclic graphs (data flows one way). RNNs, however, have loops (the hidden state from the previous step is fed into the current step). You cannot calculate gradients directly on a loop.
To solve this, BPTT unrolls the RNN into a long feedforward network.
Here is a step-by-step breakdown of how it works.

1. Unrolling the Network

Imagine an RNN processing a sentence of 5 words. Conceptually, it is one neuron with a loop. For BPTT, we visualize this as 5 distinct layers chained together.

    Time Step t=1
    t=1: Input x1
    x1​, Hidden State h1
    h1​, Output y1
    y1​.
    
    Time Step t=2
    t=2: Input x2
    x2​, Hidden State h2
    h2​ (depends on h1
    h1​), Output y2
    y2​.
    
    ...
    
    Time Step t=T
    t=T: Input xT
    xT​, Hidden State hT
    hT​, Output yT
    yT​.

Even though it looks like 5 layers, they all share the same weights. This is the crucial distinction between BPTT and standard backprop.

2. The Forward Pass
Before calculating gradients, data flows forward. At each time step t
t, the math looks like this:

    Hidden State:

    (Current input + Previous hidden state)
   
    Output:
   
    Loss:

The Total Loss () for the sequence is usually the sum of losses at every time step:

3. The Backward Pass (BPTT)
Now we need to update the weights () to minimize . We start from the last time step () and move backward to .

A. The Chain Rule Across Time
In a standard network, error flows from the output layer back to the input layer. In an RNN, error flows from the last time step back to the first time step.

To update the recurrent weight , we need to know how much the loss changed with respect to that weight. Because  was used at every time step, it contributed to the error at every time step.

Therefore, the total gradient is the sum of the gradients at each time step:


B. Flow of Gradients

When calculating the gradient for an early time step (e.g., ), the gradient depends on the future.

    The error at  affects .
    
    The error at  affects .
    
    ...
    
    The error at  affects the final Loss.

So, to find , you must chain the derivatives through all subsequent hidden states:

4. The Major Challenge: Vanishing/Exploding Gradients

Look at the chain rule equation above. It involves multiplying many matrices (Jacobians) together.

    Vanishing Gradients: If the values in these matrices are small (e.g., ), multiplying them repeatedly makes the gradient approach zero. The network stops learning long-term dependencies (it "forgets" the beginning of the sentence).
    
    Exploding Gradients: If the values are large (e.g., ), the gradient grows exponentially and becomes NaN (Not a Number), crashing the training.

Solutions:

    Gradient Clipping: If the gradient vector norm exceeds a threshold, scale it down. (Fixes exploding gradients).
    
    LSTM / GRU: Use specialized RNN architectures with "gates" that allow gradients to flow through time without being multiplied by weight matrices at every step. (Fixes vanishing gradients).
    
    ReLU Activations: Sometimes used instead of Tanh to keep gradients larger.

5. Practical Constraint: Truncated BPTT

If you have a very long sequence (e.g., a document with 1,000 words), unrolling the network for 1,000 steps is:

    Memory intensive (must store all intermediate states).
    
    Computationally slow.
    
    Prone to vanishing gradients.

Solution: We use Truncated BPTT.


We break the sequence into smaller chunks (e.g., 50 time steps). We backpropagate only through those 50 steps. The hidden state is carried over to the next chunk, but the gradient flow is cut. This treats the RNN as having a "limited memory" for gradient updates.

Summary Checklist

    Unroll: Transform the cyclic RNN graph into a linear chain of time steps.
    
    Forward: Compute hidden states and losses for all steps.
    
    Backward: Compute gradients starting from the last step () moving to the first ().
    
    Sum: Since weights are shared, sum the gradients calculated at each time step to get the total update for the weights.
    
    Update: Apply the optimizer (SGD, Adam, etc.) to update the shared weights.